In [1]:
"""Build country-level science-cooperation and UNGA-affinity data.

Required files in the same folder:
    2026_02_06_ga_voting.csv- https://digitallibrary.un.org/record/4060887?ln=en
    main file.csv containing science diplomacy data- has to be constructed using original code from https://github.com/anonreplication-wq/replication-code
"""

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


VOTES_FILE = Path("2026_02_06_ga_voting.csv")
SCIENCE_FILE = Path("main file.csv")

EU27 = set(
    "AUT BEL BGR HRV CYP CZE DNK EST FIN FRA DEU GRC HUN IRL ITA LVA LTU "
    "LUX MLT NLD POL PRT ROU SVK SVN ESP SWE".split()
)


# 1. Read 2015-2025 UNGA votes. X/non-voting observations are excluded.
votes = pd.read_csv(
    VOTES_FILE,
    usecols=["undl_id", "ms_code", "ms_name", "ms_vote", "date"],
    low_memory=False,
)
votes["date"] = pd.to_datetime(votes["date"])
votes = votes[votes["date"].between("2015-01-01", "2025-12-31")]
votes = votes[votes["ms_vote"].isin(["Y", "N", "A"])].copy()

latest_names = (
    votes.sort_values("date")
    .drop_duplicates("ms_code", keep="last")
    .set_index("ms_code")["ms_name"]
    .to_dict()
)


# 2. Match the science-cooperation CSV to UN member codes.
def clean_name(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii").upper()
    return re.sub(r"[^A-Z0-9]+", " ", text).strip()


name_to_code = {clean_name(name): code for code, name in latest_names.items()}
aliases = {
    "BRITISH VIRGIN ISLANDS": None,
    "COOK ISLANDS": None,
    "CZECH REPUBLIC": "CZE",
    "FEDERATED STATES OF MICRONESIA": "FSM",
    "KOREA REPUBLIC OF": "KOR",
    "KYRGYZ REPUBLIC": "KGZ",
    "MONTSERRAT": None,
    "NETHERLANDS": "NLD",
    "PRINCIPALITY OF LIECHTENSTEIN": "LIE",
    "TURKIYE": "TUR",
    "TURKS AND CAICOS ISLANDS": None,
    "UNKNOWN": None,
}


def country_code(country):
    name = clean_name(country)
    return aliases.get(name, name_to_code.get(name))


science = pd.read_csv(SCIENCE_FILE)
science["iso3"] = science["Country"].map(country_code)
science = science.dropna(subset=["iso3"]).copy()

country_data = (
    science.groupby("iso3")
    .agg(
        country=("Country", "first"),
        raw_entry_count=("Area_Normalized", "size"),
        cooperation_breadth=("Area_Normalized", "nunique"),
    )
    .reset_index()
)

# Binary indicators reproduce the agreement/cooperation-type tests.
sector_presence = (pd.crosstab(science["iso3"], science["Area_Normalized"]) > 0).astype(int)
sector_presence.columns = ["sector__" + column for column in sector_presence.columns]
country_data = country_data.merge(sector_presence.reset_index(), on="iso3")


# 3. Score voting affinity with a reference position.
def calculate_affinity(reference_votes):
    reference_votes = reference_votes[["undl_id", "reference_vote"]].dropna()
    pairs = votes[["undl_id", "ms_code", "ms_vote"]].merge(
        reference_votes, on="undl_id"
    )

    same_vote = pairs["ms_vote"].eq(pairs["reference_vote"])
    one_abstention = pairs["ms_vote"].eq("A") != pairs["reference_vote"].eq("A")
    pairs["score"] = np.select(
        [same_vote, one_abstention], [1.0, 0.5], default=0.0
    )

    result = (
        pairs.groupby("ms_code")["score"]
        .agg(affinity="mean", comparable_votes="size")
        .reset_index()
    )
    result["affinity_pct"] = result["affinity"] * 100
    return result[result["comparable_votes"] >= 30]


# 4. India affinity.
india_reference = votes[votes["ms_code"].eq("IND")][["undl_id", "ms_vote"]]
india_reference = india_reference.rename(columns={"ms_vote": "reference_vote"})
affinities = {"India": calculate_affinity(india_reference)}


# 5. Construct a cohesive EU position.
eu_counts = (
    votes[votes["ms_code"].isin(EU27)]
    .groupby(["undl_id", "ms_vote"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Y", "N", "A"], fill_value=0)
)
eu_participants = eu_counts.sum(axis=1)
eu_mode = eu_counts.idxmax(axis=1)
eu_cohesion = eu_counts.max(axis=1) / eu_participants
eu_reference = pd.DataFrame(
    {
        "undl_id": eu_counts.index,
        "EU": eu_mode.where((eu_participants >= 20) & (eu_cohesion >= 0.80)),
    }
).reset_index(drop=True)


# 6. Keep anchor-discriminating resolutions and calculate four affinities.
individual_anchors = (
    votes[votes["ms_code"].isin(["USA", "RUS", "CHN"])]
    .pivot(index="undl_id", columns="ms_code", values="ms_vote")
    .reset_index()
)
anchor_votes = individual_anchors.merge(eu_reference, on="undl_id", how="outer")
anchor_columns = ["USA", "RUS", "CHN", "EU"]
anchor_votes = anchor_votes[anchor_votes[anchor_columns].nunique(axis=1) >= 2]

reference_columns = {
    "United States": "USA",
    "Russia": "RUS",
    "China": "CHN",
    "European Union": "EU",
}
for reference, column in reference_columns.items():
    reference_votes = anchor_votes[["undl_id", column]].rename(
        columns={column: "reference_vote"}
    )
    affinities[reference] = calculate_affinity(reference_votes)


# 7. Merge science cooperation and affinity measures into one transparent file.
for reference, affinity in affinities.items():
    prefix = reference.replace(" ", "_")
    affinity = affinity.rename(
        columns={
            "ms_code": "iso3",
            "affinity_pct": prefix + "_affinity_pct__primary",
            "comparable_votes": prefix + "_comparable_vote_n__primary",
        }
    )
    country_data = country_data.merge(
        affinity[
            [
                "iso3",
                prefix + "_affinity_pct__primary",
                prefix + "_comparable_vote_n__primary",
            ]
        ],
        on="iso3",
        how="left",
    )

country_data.to_csv("country_level_affinities.csv", index=False)

print("Science-cooperation source rows:", len(science))
print("Matched country rows:", len(country_data))
print("Saved country_level_affinities.csv")

Science-cooperation source rows: 797
Matched country rows: 169
Saved country_level_affinities.csv


In [2]:
"""Run the reported Spearman tests on the country-level file built in step 1."""

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


data = pd.read_csv("country_level_affinities.csv")

EU27 = set(
    "AUT BEL BGR HRV CYP CZE DNK EST FIN FRA DEU GRC HUN IRL ITA LVA LTU "
    "LUX MLT NLD POL PRT ROU SVK SVN ESP SWE".split()
)
ALL_ANCHORS = EU27 | {"USA", "RUS", "CHN"}

reference_columns = {
    "India": "India_affinity_pct__primary",
    "United States": "United_States_affinity_pct__primary",
    "Russia": "Russia_affinity_pct__primary",
    "China": "China_affinity_pct__primary",
    "European Union": "European_Union_affinity_pct__primary",
}
self_codes = {
    "United States": {"USA"},
    "Russia": {"RUS"},
    "China": {"CHN"},
    "European Union": EU27,
}


def adjust_fdr(p_values):
    """Benjamini-Hochberg adjustment."""
    p_values = np.asarray(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    answer = np.empty(len(ranked))
    answer[order] = np.minimum(adjusted, 1)
    return answer


overall_rows = []
sector_rows = []
sector_columns = [column for column in data.columns if column.startswith("sector__")]


def run_tests(reference, sample_name, sample):
    affinity = reference_columns[reference]

    for predictor in ["cooperation_breadth", "raw_entry_count"]:
        rho, p = spearmanr(sample[predictor], sample[affinity])
        overall_rows.append(
            [reference, sample_name, predictor, len(sample), rho, p]
        )

    for column in sector_columns:
        present = int(sample[column].sum())
        absent = len(sample) - present
        if present >= 10 and absent >= 10:
            rho, p = spearmanr(sample[column], sample[affinity])
            sector_rows.append(
                [
                    reference,
                    sample_name,
                    column.replace("sector__", ""),
                    present,
                    absent,
                    len(sample),
                    rho,
                    p,
                ]
            )


# 1. Affinity with India: all matched science-diplomacy partners.
run_tests("India", "India affinity", data)

# 2. Retain the other anchors. 3. Remove all anchor units.
for reference in ["United States", "Russia", "China", "European Union"]:
    retained = data[~data["iso3"].isin(self_codes[reference])]
    run_tests(reference, "Other anchors retained", retained)

    excluded = data[~data["iso3"].isin(ALL_ANCHORS)]
    run_tests(reference, "All anchor units removed", excluded)


overall = pd.DataFrame(
    overall_rows,
    columns=[
        "reference",
        "sample",
        "predictor",
        "n_countries",
        "rho",
        "p_two_sided",
    ],
)
overall["p_fdr_bh"] = overall.groupby("sample")["p_two_sided"].transform(
    adjust_fdr
)

sectors = pd.DataFrame(
    sector_rows,
    columns=[
        "reference",
        "sample",
        "sector",
        "sector_present_n",
        "sector_absent_n",
        "n_countries",
        "rho",
        "p_two_sided",
    ],
)
sectors["p_fdr_bh"] = sectors.groupby(["sample", "reference"])[
    "p_two_sided"
].transform(adjust_fdr)

overall.to_csv("reproduced_overall_spearman.csv", index=False)
sectors.to_csv("reproduced_sector_spearman.csv", index=False)

print(overall.round(3).to_string(index=False))
print("\nSaved reproduced_overall_spearman.csv")
print("Saved reproduced_sector_spearman.csv")

     reference                   sample           predictor  n_countries    rho  p_two_sided  p_fdr_bh
         India           India affinity cooperation_breadth          169  0.126        0.102     0.204
         India           India affinity     raw_entry_count          169  0.090        0.247     0.247
 United States   Other anchors retained cooperation_breadth          168 -0.122        0.116     0.448
 United States   Other anchors retained     raw_entry_count          168 -0.075        0.336     0.448
 United States All anchor units removed cooperation_breadth          139 -0.179        0.035     0.135
 United States All anchor units removed     raw_entry_count          139 -0.158        0.064     0.135
        Russia   Other anchors retained cooperation_breadth          168  0.084        0.279     0.448
        Russia   Other anchors retained     raw_entry_count          168  0.040        0.605     0.605
        Russia All anchor units removed cooperation_breadth          139 